In [ ]:
import numpy as np
import pandas as pd
import nltk
import string
import re
from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')

In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.isnull().sum()

,0
review,0
sentiment,0


In [ ]:
df.duplicated().sum()

np.int64(418)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.describe()

,review,sentiment
count,49582,49582
unique,49582,2
top,No one expects the Star Trek movies to be high...,positive
freq,1,24884


In [ ]:
def removeHtml(text):
    html = re.compile(r'<.*?>')
    return html.sub(r'',text)

In [ ]:
df['review'] = df['review'].apply(removeHtml)


In [ ]:
def removePunctuations(text):
  punctuations = string.punctuation
  for char in punctuations:
    text = text.replace(char,'')
  return text

In [ ]:
df['review'] = df['review'].apply(removePunctuations)

In [ ]:
def removeStopWords(text):
  stop_words = set(stopwords.words('english'))
  words = text.split()
  filtered_words = [word for word in words if word.lower() not in stop_words]
  return ' '.join(filtered_words)

df['review'] = df['review'].apply(removeStopWords)

In [ ]:
df['review'][3]


'Basically theres family little boy Jake thinks theres zombie closet parents fighting timeThis movie slower soap opera suddenly Jake decides become Rambo kill zombieOK first youre going make film must Decide thriller drama drama movie watchable Parents divorcing arguing like real life Jake closet totally ruins film expected see BOOGEYMAN similar movie instead watched drama meaningless thriller spots3 10 well playing parents descent dialogs shots Jake ignore'

In [ ]:
X = df['review']
y = df['sentiment']

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
# text preprocessing

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train.shape

(39665,)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
cv = CountVectorizer(max_features=5000)

In [ ]:
X_train_bow = cv.fit_transform(X_train).toarray()
X_test_bow = cv.transform(X_test).toarray()

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.linear_model import LogisticRegression

In [ ]:
# Train GaussianNB
gnb = GaussianNB()
gnb.fit(X_train_bow, y_train)

# Train MultinomialNB
mnb = MultinomialNB()
mnb.fit(X_train_bow, y_train)

# Train LogisticRegression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_bow, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred_gnb = gnb.predict(X_test_bow)
y_pred_mnb = mnb.predict(X_test_bow)
y_pred_lr = lr.predict(X_test_bow)

In [ ]:
from sklearn.metrics import accuracy_score

print("GaussianNB Accuracy:", accuracy_score(y_test, y_pred_gnb))
print("MultinomialNB Accuracy:", accuracy_score(y_test, y_pred_mnb))
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

GaussianNB Accuracy: 0.7642432187153373
MultinomialNB Accuracy: 0.8456186346677422
Logistic Regression Accuracy: 0.8704245235454271


In [ ]:
import pickle

# Save the best model (Logistic Regression) and the CountVectorizer
with open('best_model_lr.pkl', 'wb') as model_file:
    pickle.dump(lr, model_file)

with open('vectorizer.pkl', 'wb') as cv_file:
    pickle.dump(cv, cv_file)

print("Model and Vectorizer exported successfully!")

Model and Vectorizer exported successfully!
